In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

ERROR 1: PROJ: proj_create_from_database: Open of /Users/sarak/.conda/envs/electrigrid-env/share/proj failed


In [2]:
os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [3]:
# load parcels only for San Diego county (will take about 5 minutes)
parcels = gpd.read_file(
    "data/Parcels_CA_2014.gdb",
    layer="CA_PARCELS_STATEWIDE",
    where="County='San Diego'").to_crs(epsg=4326)

In [4]:
# import Zillow data (make take 10-20 minutes)
fp = os.path.join('data', 'final_zillow.gpkg')
zillow = gpd.read_file(fp).to_crs(epsg=4326)

Building Footprint
There are a total of 8 parquet files that correspond to California. We select the one that contain Los Angeles county (w120_n35_w115_n30.parquet).

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
fp = os.path.join('data', 'building_parquets', 'w120_n35_w115_n30.parquet')
building = gpd.read_parquet(fp).to_crs(epsg=4326)

## Streamlined Code (takes 1 minute!)

In [7]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    how = "left",
    predicate = "intersects")

# reproject data frame to crs with meters as units
building_m = building_zillow.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# 6. keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual'], axis = 1)

multi_complete = multi_complete.drop(['index_right'], axis = 1)

multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# join back to parcels
units_multi = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(level=0)['unit_right'].sum()

# join on parcels with summed number of units
multi_summed_units = parcels_res.join(units_multi)

assert len(multi_summed_units) < len(multi_by_parcel)

# save all non-multi observations
non_multi = building_m[building_m['type'] != "Multi"].to_crs(zillow.crs)

# keep only variables of interest
non_multi = non_multi[['source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']]

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")

/Users/sarak/.conda/envs/electrigrid-env/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/tmp/ipykernel_881912/248185667.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_881912/248185667.py:70: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[po

## Final results

In [8]:
multi_summed_units.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit,unit_right
23,4241720700,San Diego,None,None,None,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",3.0,NaN
118,4241721300,San Diego,None,None,None,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",2.0,NaN
119,4241722000,San Diego,None,None,None,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79698 0.00000,...",3.0,NaN


In [9]:
multi_by_parcel.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit_left,index_right,...,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,80603,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,130.144663,498.084000
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,...,1026707.0,None,NaN,7996232.0,06073007602,h567,SDGE,RI101,140.235708,431.200368
154,4236220200,San Diego,None,None,None,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",3.0,80751,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.155171,171.388014


In [10]:
non_multi_points.head(3)

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,...,geometry,index_right,source,id,height_m,var,region,bbox,area_m2,volume_m3
7160203,Single,1959.0,3.0,None,None,None,1.0,179462.0,living,1780.0,...,POINT (-117.24993 33.38692),6701574,ms,UnitedStates_023013203_33032,3.835961,0.299546,USA,"{'xmin': -117.25014289023119, 'ymin': 33.38687...",301.592312,1156.896305
7160213,Single,1952.0,2.0,None,None,I,1.0,153443.0,living,1510.0,...,POINT (-117.24838 33.38564),6698354,ms,UnitedStates_023013203_96432,5.221156,0.925386,USA,"{'xmin': -117.24844943872387, 'ymin': 33.38562...",192.020400,1002.568394
7160303,Single,1962.0,3.0,None,None,I,2.0,401364.0,living,1540.0,...,POINT (-117.24780 33.38609),6698329,ms,UnitedStates_023013203_294280,6.511510,1.177574,USA,"{'xmin': -117.24791472273105, 'ymin': 33.38604...",237.514781,1546.579963


## Renaming and saving

In [13]:
multi_summed_units_sd = multi_summed_units.copy()

multi_by_parcel_sd = multi_by_parcel.copy()

non_multi_points_sd = non_multi_points.copy()

In [17]:
# takes a really long time :,(
multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 

In [15]:
non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

In [16]:
multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

In [14]:
## Saving! (takes at least 40 minutes)

multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 